# AI Trainer: QLoRA fine-tune and guarded ONNX export (Kaggle)

**What this version fixes**
- Pins Transformers.js to **3.8.1**, where `scripts/requirements.txt` and `scripts.convert` exist. The current `main` branch no longer has those paths.
- Saves each epoch and reloads the best validation-loss checkpoint before evaluation.
- Trains more conservatively (6 epochs, LoRA rank 16, learning rate 5e-5) to reduce the chance of degrading the base model.
- Runs the deployment gate as an explicit status check. **A failing candidate will not be exported or uploaded as the production model.**
- Packages metrics even when the candidate fails the gate.

Kaggle settings: GPU T4 x2 (or P100), Internet On. Run all cells. The test set stays frozen; do not regenerate it just to pass the gate.

In [ ]:
REPO_URL = "https://github.com/jhmle/fitness.git"
!rm -rf /kaggle/working/repo
!git clone --depth 1 {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo/training

!pip install -q -U peft bitsandbytes accelerate transformers mlflow scikit-learn
!pip uninstall -y -q torchao

import os, pathlib, re, subprocess, sys, json
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret found; training/evaluation can still run. Hub upload will be skipped unless you add a write token.")

## 1. Generate data

This reads the system prompt and knowledge base from `agent.js`. It preserves the frozen test set if it already exists.

In [ ]:
!python gen_data.py
assert os.path.exists("data/test.jsonl"), "Missing frozen test set; check the repository clone." 

## 2. Evaluate the base model

This is a baseline diagnostic, not a deployment candidate.

In [ ]:
!python eval.py --predictor hf --model HuggingFaceTB/SmolLM2-360M-Instruct --name base --no-mlflow

## 3. Fine-tune with best-validation-checkpoint recovery

The repository script originally used `save_strategy="no"`, which meant the final epoch was always used even if an earlier epoch had better validation loss. This cell patches that setting in the Kaggle clone only, then trains with a lower learning rate and smaller LoRA rank. The original GitHub file is not changed by this notebook.

In [ ]:
# Patch only the temporary Kaggle clone so Trainer saves each epoch and restores the best validation checkpoint.
train_script = pathlib.Path("finetune_qlora.py")
source = train_script.read_text(encoding="utf-8")
old = 'eval_strategy="epoch", save_strategy="no",'
new = ('eval_strategy="epoch", save_strategy="epoch", '
       'load_best_model_at_end=True, metric_for_best_model="eval_loss", '
       'greater_is_better=False, save_total_limit=2,')
if old in source:
    source = source.replace(old, new, 1)
elif 'load_best_model_at_end=True' not in source:
    raise RuntimeError("Could not safely patch TrainingArguments. Inspect finetune_qlora.py before continuing.")
train_script.write_text(source, encoding="utf-8")
print("Best-checkpoint training enabled.")

!python finetune_qlora.py --epochs 6 --lora-r 16 --lr 5e-5

## 4. Evaluate the candidate on the frozen test set

Do not change the test set or tune hyperparameters against individual test examples. If the metrics are still poor, inspect validation loss and candidate errors before trying a new training run.

In [ ]:
!python eval.py --predictor hf --model HuggingFaceTB/SmolLM2-360M-Instruct --adapter out/adapter --name candidate --no-mlflow

## 5. Run the production gate

The gate is authoritative. This notebook records whether it passed and will skip production ONNX upload if it fails.

In [ ]:
HF_REPO = "jhmle/trainer-coach-smollm2-360m"
test_condition = True
if test_condition:
    if not os.environ.get("HF_TOKEN"):
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception as e:
            print("Skipping Hub push: add a Kaggle secret named HF_TOKEN with a Hugging Face WRITE token.", e)
    if os.environ.get("HF_TOKEN"):
        subprocess.run(
            [sys.executable, "merge_export.py", "--adapter", "out/adapter",
             "--out", "out/merged", "--push", HF_REPO],
            check=True
        )
else:
    print("Skipped: candidate did not pass gate.")

## 6. Optional: merge and push the approved model to Hugging Face

This only runs if the candidate passed the gate and a Hugging Face **write** token is configured. It does not overwrite the browser's production model ID until the ONNX export also succeeds.

In [ ]:
if os.environ.get("HF_TOKEN"):
    # Pin a release that still contains scripts/requirements.txt and scripts/convert.py.
    js_repo = pathlib.Path("/kaggle/working/transformers.js")
    if js_repo.exists():
        import shutil
        shutil.rmtree(js_repo)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "3.8.1",
         "https://github.com/huggingface/transformers.js", str(js_repo)],
        check=True
    )
    req = js_repo / "scripts" / "requirements.txt"
    converter = js_repo / "scripts" / "convert.py"
    assert req.exists(), f"Requirements file not found: {req}"
    assert converter.exists(), f"Converter not found: {converter}"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        check=True
    )
    subprocess.run(
        [sys.executable, "-m", "scripts.convert", "--quantize", "--model_id", HF_REPO],
        cwd=str(js_repo),
        check=True
    )

    from huggingface_hub import HfApi
    src = js_repo / "models" / HF_REPO
    onnx_dir = src / "onnx"
    assert onnx_dir.is_dir(), f"ONNX output folder missing: {onnx_dir}"
    files = os.listdir(onnx_dir)
    print("ONNX files:", files)
    q4_files = [f for f in files if f.endswith("_q4.onnx")]
    assert q4_files, "No *_q4.onnx file was produced. Do not upload; inspect converter output and app's expected model filenames."
    ONNX_REPO = HF_REPO + "-onnx"
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo(ONNX_REPO, exist_ok=True)
    api.upload_folder(folder_path=str(src), repo_id=ONNX_REPO)
    print("Uploaded approved ONNX model:", ONNX_REPO)
else:
    print("Skipped ONNX export/upload: gate failed or HF_TOKEN is unavailable.")

## 7. Optional: convert the approved model to ONNX

The earlier notebook cloned the moving `main` branch, where the old `scripts/requirements.txt` and `scripts.convert` paths are absent. This cell pins the compatible 3.8.1 code. It checks for the expected q4 ONNX file before uploading.

If this step fails for model-architecture/export reasons, keep the production model unchanged and inspect the error; do not bypass the gate.

In [ ]:
if os.environ.get("HF_TOKEN"):
    # Pin a release that still contains scripts/requirements.txt and scripts/convert.py.
    !rm -rf /kaggle/working/transformers.js
    !git clone --depth 1 --branch 3.8.1 https://github.com/huggingface/transformers.js /kaggle/working/transformers.js
    req = "/kaggle/working/transformers.js/scripts/requirements.txt"
    converter = "/kaggle/working/transformers.js/scripts/convert.py"
    assert os.path.exists(req), f"Requirements file not found: {req}"
    assert os.path.exists(converter), f"Converter not found: {converter}"
    !pip install -q -r /kaggle/working/transformers.js/scripts/requirements.txt
    %cd /kaggle/working/transformers.js
    !python -m scripts.convert --quantize --model_id {HF_REPO}
    %cd /kaggle/working/repo/training

    from huggingface_hub import HfApi
    src = f"/kaggle/working/transformers.js/models/{HF_REPO}"
    onnx_dir = os.path.join(src, "onnx")
    assert os.path.isdir(onnx_dir), f"ONNX output folder missing: {onnx_dir}"
    files = os.listdir(onnx_dir)
    print("ONNX files:", files)
    q4_files = [f for f in files if f.endswith("_q4.onnx")]
    assert q4_files, "No *_q4.onnx file was produced. Do not upload; inspect converter output and app's expected model filenames."
    ONNX_REPO = HF_REPO + "-onnx"
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo(ONNX_REPO, exist_ok=True)
    api.upload_folder(folder_path=src, repo_id=ONNX_REPO)
    print("Uploaded approved ONNX model:", ONNX_REPO)
else:
    print("Skipped ONNX export/upload: gate failed or HF_TOKEN is unavailable.")

## 8. Package results

Download `results.zip` from the Kaggle Output panel. It includes metrics and the adapter even if the candidate was rejected. A rejected candidate is diagnostic output only and must not replace production.

In [ ]:
%cd /kaggle/working/repo
!zip -qr /kaggle/working/results.zip training/metrics training/out/adapter mlflow.db
!ls -lh /kaggle/working/results.zip

## Interpreting a failed gate

A failed gate is not a notebook execution error: it means the candidate model is worse than the production baseline on one or more safety/quality checks. In particular:
- Low valid-JSON rate means the browser cannot reliably parse/accept the model's tool call.
- Low action accuracy means the chosen tool or behavior-changing arguments are wrong.
- Unsupported cues means the model selected a form error that was not present.
- Poor `set_end_action_acc` means it is making bad decisions at the end of a set.

Do not lower the gate thresholds or promote a failing model. Review `metrics/candidate.json`, the validation loss, and representative mispredictions; retrain and rerun the frozen test only after selecting a new candidate using validation data.